In [1]:
library(tidyverse)  # Data wrangling packages.
library(rtracklayer)
library(GenomicRanges)
library(stringr)

my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: GenomicRanges

Loading required package: stats4

Loading required package: BiocGenerics

Loading required package: generics


Attaching package: ‘generics’


The following object is masked from ‘package:lubridate’:

    as.difftime


The following object is masked from ‘package:dplyr’:

    explain


The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



In [ ]:
for (i in 1:22) {
  command <- paste0("gsutil -u ", Sys.getenv("GOOGLE_PROJECT"), " cp gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr", i, ".bim gs://fc-secure-5e7b1abd-1cd7-485e-8567-bf3cb4814b5c/acaf_threshold.chr", i, ".bim")
  system(command) 
    
}

In [46]:
process_file <- function(name_of_file_in_bucket) {
    # Download the file from the cloud bucket
    system(paste0("gsutil cp ", my_bucket, "/weights/hg38/", name_of_file_in_bucket, " ."), intern=T)
    
    # Read the weights data from the file
    weights <- read.csv(name_of_file_in_bucket)
    
    # In AoU SNPIDs are read as strings so you need them in both orders because we don't know how they will appear
    new1 <- weights
    new1$SNPID <- paste0(new1$chr, ":", new1$pos, ":", new1$a1, ":", new1$a2)
    new2 <- weights
    new2$SNPID <- paste0(new2$chr, ":", new2$pos, ":", new2$a2, ":", new2$a1)
    
    # Split the repeated file into singletons and indels
    split_df <- split(new2, ifelse(nchar(new2$a1) > 1 | nchar(new2$a2) > 1, "indel", "singleton"))
    
    # Ensure 'indel' part exists, if not, set it to NULL
    if ("indel" %in% names(split_df)) {
        new2_indel <- split_df$indel
    } else {
        new2_indel <- NULL
    }
    
    # Check if new2_indel has nonzero length and is not NULL
    if (!is.null(new2_indel) && nrow(new2_indel) > 0) {
        # Create a vector to keep track of rows to remove
        rows_to_remove <- logical(nrow(new2_indel))
        
        for (i in 1:nrow(new2_indel)) {   
            print(paste("Processing row", i))  
            
            chromosome <- new2_indel$chr[i]
            position <- new2_indel$pos[i]
            a1 <- new2_indel$a1[i]
            
            # Define the command string
            command <- paste0("gsutil cat gs://fc-secure-5e7b1abd-1cd7-485e-8567-bf3cb4814b5c/acaf_threshold.", chromosome, ".bim | grep -P '", position, "'")
            
            # Execute the command
            output <- system(command, intern = TRUE)  # Replace spaces with tabs
            
            # Split the output string by newline
            lines <- unlist(strsplit(output, "\n"))
            
            if (length(lines) == 0) {
                print(paste("No valid output for position", position, "- skipping to next row."))
                next  # Skip to the next iteration
            }
            
            # Initialize an empty list to store the data
            data <- list()
            
            # Iterate over each line
            for (line in lines) {
                # Split the line by tab delimiter
                values <- unlist(strsplit(line, "\t"))
                # Append the values to the list
                data[[length(data) + 1]] <- values
            }
            
            # Convert the list to a dataframe
            df <- as.data.frame(do.call(rbind, data))
            
            # Print the value of a1 for debugging
            print(a1)
            
            # Check if a1 is in V5 and V6
            if (a1 %in% df$V5 & a1 %in% df$V6) {
                print("Remove")
                rows_to_remove[i] <- TRUE  # Mark row for removal
            } else {
                print("No")
                rows_to_remove[i] <- FALSE  # Keep the row
            }
        }
    } else {
        print("new2_indel is empty or NULL, skipping removal process.")
    }
    
    # Remove rows after the loop
    if (!is.null(new2_indel)) {
        new3 <- new2_indel[!rows_to_remove, ]
    } else {
        new3 <- data.frame()  # Create an empty dataframe if new2_indel is NULL
    }
    
    # Print final result
    print(new3)
    
    # Merge all of our weight rows
    merged_df2 <- rbind(new1, split_df$singleton)
    merged_df2 <- rbind(merged_df2, new3)
    merged_df2 <- merged_df2[, -2]  # Remove second column
    head(merged_df2)
    dim(merged_df2)
    
    # Write the output file
    destination_filename <- name_of_file_in_bucket
    write_delim(merged_df2, destination_filename, delim = " ", quote = "none")
    
    # Upload the processed file to the bucket
    system(paste0("gsutil cp ./", destination_filename, " ", my_bucket, "/weights/hg38/"), intern=T)
    
    print(paste("File", destination_filename, "processed and uploaded successfully."))
}

# Define the list of filenames to process
file_list <- c("asthma_PRS_weights.txt",
                "atrial_fibrilation_PRS_weights.txt",
                "breast_cancer_PRS_weights.txt",
                "ckd_PRS_weights.txt",
                "coronary_heart_disease_PRS_weights.txt",
                "hypercholesterolemia_PRS_weights.txt",
                "prostate_cancer_PRS_weights.txt",
                "t1d_PRS_weights.txt",
                "t2d_PRS_weights.txt")

# Loop through the file list and call the process_file function for each
lapply(file_list, process_file)


[1] "new2_indel is empty or NULL, skipping removal process."
data frame with 0 columns and 0 rows
[1] "File asthma_PRS_weights.txt processed and uploaded successfully."
[1] "new2_indel is empty or NULL, skipping removal process."
data frame with 0 columns and 0 rows
[1] "File atrial_fibrilation_PRS_weights.txt processed and uploaded successfully."
[1] "Processing row 1"
[1] "C"
[1] "Remove"
[1] "Processing row 2"


Warning message in system(command, intern = TRUE):
“running command 'gsutil cat gs://fc-secure-5e7b1abd-1cd7-485e-8567-bf3cb4814b5c/acaf_threshold.chr1.bim | grep -P '145830811'' had status 1”


[1] "No valid output for position 145830811 - skipping to next row."
[1] "Processing row 3"
[1] "C"
[1] "Remove"
[1] "Processing row 4"
[1] "TA"
[1] "Remove"
[1] "Processing row 5"
[1] "TTCTGAAACAGGG"
[1] "No"
[1] "Processing row 6"
[1] "T"
[1] "Remove"
[1] "Processing row 7"
[1] "C"
[1] "Remove"
[1] "Processing row 8"
[1] "G"
[1] "Remove"
[1] "Processing row 9"
[1] "CT"
[1] "Remove"
[1] "Processing row 10"
[1] "C"
[1] "Remove"
[1] "Processing row 11"
[1] "CT"
[1] "Remove"
[1] "Processing row 12"
[1] "AT"
[1] "No"
[1] "Processing row 13"
[1] "TTG"
[1] "Remove"
[1] "Processing row 14"
[1] "AAT"
[1] "Remove"
[1] "Processing row 15"
[1] "C"
[1] "No"
[1] "Processing row 16"
[1] "T"
[1] "Remove"
[1] "Processing row 17"
[1] "T"
[1] "Remove"
[1] "Processing row 18"
[1] "AG"
[1] "No"
[1] "Processing row 19"
[1] "T"
[1] "No"
[1] "Processing row 20"
[1] "GT"
[1] "Remove"
[1] "Processing row 21"
[1] "CA"
[1] "Remove"
[1] "Processing row 22"
[1] "CT"
[1] "Remove"
[1] "Processing row 23"
[1] "G"
[1

Warning message in system(command, intern = TRUE):
“running command 'gsutil cat gs://fc-secure-5e7b1abd-1cd7-485e-8567-bf3cb4814b5c/acaf_threshold.chr3.bim | grep -P '170365752'' had status 1”


[1] "No valid output for position 170365752 - skipping to next row."
[1] "Processing row 7"
[1] "T"
[1] "Remove"
[1] "Processing row 8"
[1] "C"
[1] "Remove"
[1] "Processing row 9"
[1] "G"
[1] "Remove"
[1] "Processing row 10"
[1] "TG"
[1] "No"
[1] "Processing row 11"
[1] "T"
[1] "Remove"
[1] "Processing row 12"
[1] "C"
[1] "No"
[1] "Processing row 13"
[1] "C"
[1] "Remove"
[1] "Processing row 14"
[1] "T"
[1] "No"
[1] "Processing row 15"
[1] "T"
[1] "Remove"
[1] "Processing row 16"
[1] "C"
[1] "Remove"
[1] "Processing row 17"
[1] "G"
[1] "No"
[1] "Processing row 18"
[1] "CT"
[1] "Remove"
[1] "Processing row 19"
[1] "C"
[1] "Remove"
[1] "Processing row 20"
[1] "TGGCAGTGGCAGC"
[1] "No"
[1] "Processing row 21"
[1] "A"
[1] "No"
[1] "Processing row 22"


Warning message in system(command, intern = TRUE):
“running command 'gsutil cat gs://fc-secure-5e7b1abd-1cd7-485e-8567-bf3cb4814b5c/acaf_threshold.chrX.bim | grep -P '51502424'' had status 1”


[1] "No valid output for position 51502424 - skipping to next row."
                                id effect_allele     weight   chr       pos
16          chr1:183063312:C:CTAAG         CTAAG 0.04132373  chr1 183063312
65         chr3:170365752:C:CTTTTT             C 0.10473957  chr3 170365752
105            chr6:160160511:T:TG            TG 0.19078190  chr6 160160511
120          chr8:26036685:C:CCAAA             C 0.07704844  chr8  26036685
143             chr9:79475808:T:TG             T 0.07842624  chr9  79475808
227         chr17:71121391:G:GTTAT             G 0.15754807 chr17  71121391
246 chr20:62426005:T:TGGCAGTGGCAGC TGGCAGTGGCAGC 0.04463005 chr20  62426005
256            chr22:28695868:A:AG             A 0.39020562 chr22  28695868
265             chrX:51502424:G:GA            GA 0.09970145  chrX  51502424
               a1     a2                          SNPID
16          CTAAG      C         chr1:183063312:C:CTAAG
65              C CTTTTT        chr3:170365752:CTTTTT:C
105 

[[1]]
[1] "File asthma_PRS_weights.txt processed and uploaded successfully."

[[2]]
[1] "File atrial_fibrilation_PRS_weights.txt processed and uploaded successfully."

[[3]]
[1] "File breast_cancer_PRS_weights.txt processed and uploaded successfully."

[[4]]
[1] "File ckd_PRS_weights.txt processed and uploaded successfully."

[[5]]
[1] "File coronary_heart_disease_PRS_weights.txt processed and uploaded successfully."

[[6]]
[1] "File hypercholesterolemia_PRS_weights.txt processed and uploaded successfully."

[[7]]
[1] "File prostate_cancer_PRS_weights.txt processed and uploaded successfully."

[[8]]
[1] "File t1d_PRS_weights.txt processed and uploaded successfully."

[[9]]
[1] "File t2d_PRS_weights.txt processed and uploaded successfully."